<a href="https://colab.research.google.com/github/afisla/ssh-colab/blob/main/Kodexplorer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =============================================================================
# COMBINED COLAB SCRIPT: SSH Server + KODExplorer File Manager (Afisla Tunnel)
# =============================================================================
# Jalankan cell ini di Google Colab untuk menginstal dan mengaktifkan:
# 1. SSH Server (Port 2222) via Afisla Tunnel
# 2. KODExplorer Web File Manager (Port 8008) via Afisla Tunnel (phpfile)
# =============================================================================

import os
import sys
import time
import re
import glob
import json
import threading
import datetime
import subprocess
import urllib.request
from IPython.display import display, HTML

# -----------------------------------------------------------------------------
# KONFIGURASI GLOBAL
# -----------------------------------------------------------------------------
SSH_PORT = 2222
ROOT_PASSWORD = "Zavin123"
KODEXPLORER_PORT = 8008
KODEXPLORER_DIR = "/content/kodexplorer"
KODEXPLORER_DOMAIN = "phpfile"

AFISLA_BIN = "/usr/local/bin/afisla"
AFISLA_SSH_LOG = "afisla_ssh.log"
AFISLA_KOD_LOG = "afisla_kod.log"

# Variabel State Proses
sshd_process = None
php_process = None
afisla_ssh_process = None
afisla_kod_process = None
current_relay_port = None


def cleanup_old_sessions():
    """Membersihkan sesi, log, dan proses lama yang masih berjalan."""
    print("\n[CLEANUP] Menghentikan semua layanan lama...")
    services = ["cloudflared", "afisla", "lt", "serveo", "sshd", "php"]
    for s in services:
        subprocess.run(["pkill", "-f", s], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    logs = [AFISLA_SSH_LOG, AFISLA_KOD_LOG, "cf.log", "sshd_config_custom"]
    for l in logs:
        if os.path.exists(l):
            try:
                os.remove(l)
            except Exception:
                pass
    time.sleep(2)
    print("[OK] Sesi lama berhasil dibersihkan.")


def install_dependencies():
    """Memeriksa dan menginstal paket sistem yang dibutuhkan (PHP, SSH, Git, Node.js, Afisla)."""
    print("\n[+] Memeriksa dan menginstal dependensi sistem...")

    # Install paket via apt-get
    apt_packages = [
        "openssh-server", "curl", "git", "unzip",
        "php", "php-cli", "php-zip", "php-mbstring", "php-xml", "php-curl", "php-gd"
    ]

    needed_pkgs = []
    for pkg in apt_packages:
        res = subprocess.run(["dpkg", "-s", pkg], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        if res.returncode != 0:
            needed_pkgs.append(pkg)

    if needed_pkgs:
        print(f"[+] Menginstal paket APT: {', '.join(needed_pkgs)}")
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "-qq"] + needed_pkgs, check=True)

    # Install Afisla Tunnel Client (bila belum ada)
    if not os.path.exists(AFISLA_BIN):
        print("[+] Menginstal Afisla Tunnel Client Binary...")
        afisla_url = "https://github.com/afisla/tunnel/releases/download/v0.4.0/afisla-linux-amd64"
        subprocess.run(["curl", "-fsSLo", AFISLA_BIN, afisla_url], check=True)
        subprocess.run(["chmod", "+x", AFISLA_BIN], check=True)

    # Setup NVM & Opencode-AI
    if subprocess.run(["which", "opencode-ai"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode != 0:
        print("[+] Menginstal NVM & opencode-ai...")
        os.system("curl -o- https://raw.githubusercontent.com/nvm-sh/nvm/v0.40.6/install.sh | bash > /dev/null 2>&1")
        os.system("source /root/.bashrc")
        os.system('bash -c \'export NVM_DIR="$HOME/.nvm"; [ -s "$NVM_DIR/nvm.sh" ] && . "$NVM_DIR/nvm.sh" && [ -s "$NVM_DIR/bash_completion" ] && . "$NVM_DIR/bash_completion" && nvm install 24 && nvm use 24 > /dev/null 2>&1 && npm install -g opencode-ai@latest > /dev/null 2>&1\'')

    # Atur Zona Waktu ke Asia/Jakarta
    os.system("ln -sf /usr/share/zoneinfo/Asia/Jakarta /etc/localtime")
    print("[OK] Semua dependensi siap.")


def setup_kodexplorer():
    """Mengunduh KODExplorer jika belum ada."""
    print("\n[+] Mengonfigurasi KODExplorer...")
    if not os.path.exists(KODEXPLORER_DIR):
        print(f"[+] Cloning KODExplorer ke {KODEXPLORER_DIR}...")
        subprocess.run(["git", "clone", "https://github.com/kalcaddle/KODExplorer.git", KODEXPLORER_DIR], check=True)
    print(f"[OK] KODExplorer SIAP di {KODEXPLORER_DIR}")


def start_php_server():
    """Menjalankan server PHP built-in untuk KODExplorer."""
    global php_process
    print(f"[+] Menjalankan PHP Server di port {KODEXPLORER_PORT}...")

    #index_path = os.path.join(KODEXPLORER_DIR, "index.php")
    php_process = subprocess.Popen(
        ["php", "-S", f"0.0.0.0:{KODEXPLORER_PORT}", "-t", KODEXPLORER_DIR],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )
    time.sleep(2)

    # Verifikasi server berjalan
    try:
        res = urllib.request.urlopen(f"http://localhost:{KODEXPLORER_PORT}/")
        print(f"[OK] PHP Server aktif (Status Response: {res.status}).")
    except Exception as e:
        print(f"[!] Peringatan verifikasi PHP Server: {e}")


def setup_ssh_server():
    """Mengatur kata sandi root dan menjalankan daemon SSH Server."""
    global sshd_process
    print(f"\n[+] Mengonfigurasi SSH Server pada port {SSH_PORT}...")

    # Set root password
    p = subprocess.Popen(["echo", f"root:{ROOT_PASSWORD}"], stdout=subprocess.PIPE)
    subprocess.run(["chpasswd"], stdin=p.stdout, check=True)
    p.stdout.close()

    # Buat konfigurasi sshd custom
    ssh_config_content = f"""Port {SSH_PORT}
PermitRootLogin yes
PasswordAuthentication yes
PubkeyAuthentication no
ChallengeResponseAuthentication no
UsePAM yes
LogLevel VERBOSE
ClientAliveInterval 30
ClientAliveCountMax 3
TCPKeepAlive yes
Subsystem sftp /usr/lib/openssh/sftp-server
"""
    config_file = os.path.abspath("sshd_config_custom")
    with open(config_file, "w") as f:
        f.write(ssh_config_content)

    # Buat direktori runtime sshd & regenerasi host keys
    os.makedirs("/var/run/sshd", exist_ok=True)
    os.makedirs("/run/sshd", exist_ok=True)
    subprocess.run(["chmod", "0755", "/var/run/sshd"], check=False)

    for key_file in glob.glob("/etc/ssh/ssh_host_*"):
        try:
            os.remove(key_file)
        except Exception:
            pass
    subprocess.run(["ssh-keygen", "-A"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)

    # Jalankan OpenSSH Server
    sshd_process = subprocess.Popen(['/usr/sbin/sshd', '-D', '-f', config_file],
                                    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(2)
    print(f"[OK] SSH Server berjalan di port {SSH_PORT} (Password Root: {ROOT_PASSWORD}).")


def start_tunnels():
    """Menjalankan tunnel Afisla untuk SSH dan KODExplorer."""
    global afisla_ssh_process, afisla_kod_process
    print("\n[+] Menjalankan Afisla Tunnels...")

    # 1. Tunnel untuk SSH (port local 2222)
    ssh_log_file = open(AFISLA_SSH_LOG, "w")
    afisla_ssh_process = subprocess.Popen(
        [AFISLA_BIN, "client", "--port-local", str(SSH_PORT)],
        stdout=ssh_log_file,
        stderr=subprocess.STDOUT
    )

    # 2. Tunnel untuk KODExplorer (port local 8008, domain custom 'phpfile')
    kod_log_file = open(AFISLA_KOD_LOG, "w")
    afisla_kod_process = subprocess.Popen(
        [AFISLA_BIN, "client", "--port-local", str(KODEXPLORER_PORT), "--domain", KODEXPLORER_DOMAIN],
        stdout=kod_log_file,
        stderr=subprocess.STDOUT
    )
    print("[OK] Proses tunnel Afisla berhasil dimulai.")


def monitor_and_display():
    """Thread monitoring untuk mendeteksi log tunnel, menampilkan UI, dan auto-restart."""
    global sshd_process, php_process, afisla_ssh_process, afisla_kod_process, current_relay_port
    config_file = os.path.abspath("sshd_config_custom")
    ui_rendered = False

    while True:
        time.sleep(8)

        # Cegah Colab dari masalah buffer stdout
        sys.stdout.write('\x00')
        sys.stdout.flush()

        # Monitor SSH process
        if sshd_process and sshd_process.poll() is not None:
            sshd_process = subprocess.Popen(['/usr/sbin/sshd', '-D', '-f', config_file],
                                            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

        # Monitor PHP process
        if php_process and php_process.poll() is not None:
            index_path = os.path.join(KODEXPLORER_DIR, "index.php")
            php_process = subprocess.Popen(
                ["php", "-S", f"0.0.0.0:{KODEXPLORER_PORT}", "-t", KODEXPLORER_DIR, index_path],
                stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
            )

        # Cek Log SSH Afisla Tunnel untuk Relay Port
        if os.path.exists(AFISLA_SSH_LOG):
            with open(AFISLA_SSH_LOG, "r") as f:
                log = f.read()
                match = re.search(r"(?:port|relay)[:\s]+(\d{4,5})", log, re.IGNORECASE) or re.search(r"(\d{5})", log)
                if match:
                    new_port = match.group(1)
                    if new_port != current_relay_port or not ui_rendered:
                        current_relay_port = new_port
                        ui_rendered = True

                        ssh_cmd = f"ssh -o StrictHostKeyChecking=no -o UserKnownHostsFile=/dev/null -o ServerAliveInterval=30 -o PreferredAuthentications=password -o ProxyCommand='nc relay.afisla.web.id {current_relay_port}' root@127.0.0.1 -p {SSH_PORT}"
                        uid = str(int(time.time() * 1000))

                        kod_url = f"https://{KODEXPLORER_DOMAIN}.afisla.web.id"

                        print("\n" + "=" * 70)
                        print(" [AFISLA TUNNELS ACTIVE]")
                        print(f" SSH User       : root")
                        print(f" SSH Password   : {ROOT_PASSWORD}")
                        print(f" SSH Relay Port : {current_relay_port}")
                        print(f" KODExplorer URL: {kod_url}")
                        print("=" * 70)

                        # Tampilan HTML Dashboard di Notebook Colab
                        display(HTML(f"""
                        <div style="background:#1e1e2e; border-radius:12px; padding:20px; margin:15px 0; border:1px solid #313244; font-family:sans-serif;">
                            <h3 style="color:#a6e3a1; margin-top:0; border-bottom:1px solid #45475a; padding-bottom:10px;">
                                🚀 Services Ready & Online
                            </h3>

                            <!-- KODExplorer Card -->
                            <div style="background:#181825; border-radius:8px; padding:12px; margin-bottom:15px; border:1px solid #45475a;">
                                <div style="color:#89b4fa; font-weight:bold; font-size:14px; margin-bottom:4px;">
                                    📁 KODExplorer Web File Manager
                                </div>
                                <div style="color:#cdd6f4; font-size:13px;">
                                    Akses URL: <a href="{kod_url}" target="_blank" style="color:#f9e2af; font-weight:bold; text-decoration:underline;">{kod_url}</a>
                                </div>
                            </div>

                            <!-- SSH Terminal Card -->
                            <div style="background:#181825; border-radius:8px; padding:12px; border:1px solid #45475a;">
                                <div style="color:#f38ba8; font-weight:bold; font-size:14px; margin-bottom:6px;">
                                    🔑 SSH Connection Command (Port Relay: {current_relay_port})
                                </div>
                                <textarea id="cmd_{uid}" readonly style="width:100%; background:#11111b; color:#a6e3a1; border:1px solid #313244; border-radius:6px; padding:10px; font-family:monospace; font-size:12px; resize:none; outline:none;" rows="2" onclick="this.select()">{ssh_cmd}</textarea>
                                <button onclick="var t=document.getElementById('cmd_{uid}');t.select();document.execCommand('copy');this.innerText='Copied!';var b=this;setTimeout(function(){{b.innerText='Copy Command'}},1500)" style="margin-top:8px; background:#f38ba8; color:#1e1e2e; border:none; padding:6px 16px; border-radius:6px; cursor:pointer; font-weight:bold; font-size:13px;">Copy Command</button>
                            </div>
                        </div>
                        """))


if __name__ == "__main__":
    cleanup_old_sessions()
    install_dependencies()
    setup_kodexplorer()
    start_php_server()
    setup_ssh_server()
    start_tunnels()

    # Jalankan thread monitoring di background
    threading.Thread(target=monitor_and_display, daemon=True).start()

    print("\n[+] Semua layanan berjalan. Menunggu tunnel terhubung...")

    # Loop utama untuk menjaga sesi Colab tidak terputus (Heartbeat)
    try:
        while True:
            time.sleep(30)
            now = datetime.datetime.now().strftime('%H:%M:%S')
            print(f"[{now}] Heartbeat: Layanan SSH & KODExplorer tetap aktif...")
    except KeyboardInterrupt:
        print("\n[!] Menghentikan semua layanan...")
        cleanup_old_sessions()
        print("[OK] Selesai.")


[CLEANUP] Menghentikan semua layanan lama...
[OK] Sesi lama berhasil dibersihkan.

[+] Memeriksa dan menginstal dependensi sistem...
[+] Menginstal paket APT: php, php-cli, php-zip, php-mbstring, php-xml, php-curl, php-gd
[+] Menginstal Afisla Tunnel Client Binary...
[+] Menginstal NVM & opencode-ai...
[OK] Semua dependensi siap.

[+] Mengonfigurasi KODExplorer...
[+] Cloning KODExplorer ke /content/kodexplorer...
[OK] KODExplorer SIAP di /content/kodexplorer
[+] Menjalankan PHP Server di port 8008...
[OK] PHP Server aktif (Status Response: 200).

[+] Mengonfigurasi SSH Server pada port 2222...
[OK] SSH Server berjalan di port 2222 (Password Root: Zavin123).

[+] Menjalankan Afisla Tunnels...
[OK] Proses tunnel Afisla berhasil dimulai.

[+] Semua layanan berjalan. Menunggu tunnel terhubung...
 
 [AFISLA TUNNELS ACTIVE]
 SSH User       : root
 SSH Password   : Zavin123
 SSH Relay Port : 30104
 KODExplorer URL: https://phpfile.afisla.web.id


  [14:04:04] Heartbeat: Layanan SSH & KODExplorer tetap aktif...
    [14:04:34] Heartbeat: Layanan SSH & KODExplorer tetap aktif...
    [14:05:04] Heartbeat: Layanan SSH & KODExplorer tetap aktif...
   [14:05:34] Heartbeat: Layanan SSH & KODExplorer tetap aktif...
    [14:06:04] Heartbeat: Layanan SSH & KODExplorer tetap aktif...
    [14:06:34] Heartbeat: Layanan SSH & KODExplorer tetap aktif...
    [14:07:04] Heartbeat: Layanan SSH & KODExplorer tetap aktif...
   [14:07:34] Heartbeat: Layanan SSH & KODExplorer tetap aktif...
    [14:08:04] Heartbeat: Layanan SSH & KODExplorer tetap aktif...
    [14:08:34] Heartbeat: Layanan SSH & KODExplorer tetap aktif...
    [14:09:04] Heartbeat: Layanan SSH & KODExplorer tetap aktif...
   [14:09:34] Heartbeat: Layanan SSH & KODExplorer tetap aktif...
    [14:10:04] Heartbeat: Layanan SSH & KODExplorer tetap aktif...
    [14:10:34] Heartbeat: Layanan SSH & KODExplorer tetap aktif...
    [14:11:04] Heartbeat: Layanan SSH & KODExplorer tetap aktif...
